# Storage precision: quantize the index, keep the answer — retrieve→rescore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/22-precision/precision.ipynb)

Built from [`cookbook/book/chapters/22-precision/precision.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/22-precision/precision.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** a deployment-default `storage_precision` (`f32` / `f16` / `int8`) on the ANN
sidecar index, set through `jammi.connect(..., config=...)` · the two-stage
retrieve→rescore path (`k * oversample` quantized candidates, exact-`f32` re-ranked) ·
the per-request `oversample` override, set through `search(..., oversample=...)` ·
**Theory:** approximate nearest-neighbour search and the recall-vs-cost trade
(Johnson et al. 2021), HNSW as the navigable proximity graph the quantized index is built
over (Malkov & Yashunin 2020) · **Rail:** measurement (recall@10 against the engine's own exact search, and the
sidecar's on-disk byte count, both measured live).

Every embedding table so far in this book keeps its ANN sidecar at full `f32` precision:
exact, single-stage search. That is the right default, but it is not the only point on the
recall-vs-memory curve. The engine's ANN sidecar can instead store its HNSW graph's own
vectors at a **quantized** `storage_precision` — `f16` or `int8` — which shrinks the graph a
search actually traverses, at the cost of making the graph's own stored vectors lossy. A
quantized table therefore searches in **two stages**: the quantized graph proposes
`k * oversample` candidates, and each candidate is re-scored by an exact cosine read off a
raw-`f32` companion kept alongside the graph — so the *answer* a caller receives is computed
from exact vectors, never from the lossy quantized ones. `f32` tables skip this entirely
(single-stage, no companion). This chapter embeds the papers once, builds `f32` and `int8` tables over the
same vectors, measures their recall against the exact top-*k*, measures their sidecars' sizes,
and dials `oversample` to show the recall/cost trade the knob buys — every knob through the
same public `jammi` front door every other chapter uses.

## The corpus and the held-out queries

Every tenth paper is a query, held out of the corpus, so a query's nearest neighbours are
other papers — recall@1 is a real number, not the trivial self-match. The papers are embedded
**once**, into an `f32` table; every table below imports those same vectors
(`import_embeddings`, no encoder run), so the only thing that varies between tables is the
precision. The ground truth each recall is scored against is `search(exact=True)` on the `f32`
table: it scores every stored vector, so it returns the true top-`k` whatever an index holds.

In [ ]:
from jammi_cookbook import contracts, precision, scale

SCALE = scale.current()
K = precision.K
corpus = precision.corpus(SCALE)
print(f"corpus {len(corpus.corpus_ids):,} papers × {corpus.dims} dims, "
      f"{len(corpus.queries)} held-out queries")

`storage_precision` is a deployment value, set on the session through `connect(config=...)`
and stamped on every table the session creates, so comparing `int8` with `f32` means two
sessions, not one with a per-call switch: `precision.built(corpus, precision, oversample)`
opens one and imports the vectors.

## The `f32` baseline — exact, single-stage

In [ ]:
with precision.built(corpus, "f32", 4) as (db, table):
    recall_f32 = precision.recall(db, corpus)
    f32_bytes = precision.bundle_bytes(db, table)
print(f"f32 recall@{K}: {recall_f32:.4f}")
print(f"f32 sidecar: {f32_bytes}")

In [ ]:
assert "rawf32" not in f32_bytes, "an f32 index writes no rescore companion"
contracts.assert_close("precision.f32_recall_at_10", recall_f32, tol=0.01)

## `int8` at the default oversample — the recall-vs-memory headline

The default oversample (`4`) retrieves `k * 4` candidates from the quantized graph before
the exact-`f32` rescore truncates back to `k`. Does quantizing the graph's own vectors — and
shrinking the structure a search actually traverses — cost recall once the rescore runs?

In [ ]:
with precision.built(corpus, "int8", 4) as (db, table):
    recall_int8 = precision.recall(db, corpus)
    int8_bytes = precision.bundle_bytes(db, table)
graph_ratio = int8_bytes["usearch"] / f32_bytes["usearch"]
print(f"int8 recall@{K} (oversample=4): {recall_int8:.4f}   (f32: {recall_f32:.4f})")
print(f"int8 graph (.usearch): {int8_bytes['usearch']:,} bytes  "
      f"({graph_ratio:.1%} of the f32 graph)")
print(f"int8 rescore companion (.rawf32): {int8_bytes['rawf32']:,} bytes")

In [ ]:
contracts.assert_close("precision.int8_os4_recall_at_10", recall_int8, tol=0.01)
contracts.assert_close("precision.int8_vs_f32_graph_ratio", graph_ratio, tol=0.03)
assert graph_ratio < 1.0, "an int8 graph is smaller than an f32 one"
if SCALE is scale.Scale.FULL:
    assert graph_ratio < 0.5, "at a real embedding width the vectors dominate the graph"

The two-stage design at work: the graph a search traverses is small and lossy, but the
*answer* is re-derived from exact vectors before it is returned. How much smaller depends
on the vector width: a graph stores each vector and its links, and quantization shrinks
only the vector — at a real embedding's hundreds of dimensions the vectors dominate and
the graph falls to about a quarter; at the fixture encoder's 32 the links weigh as much.

### The honest complication: the rescore companion is not free

The `.usearch` graph shrinks — the structure held, and for a hot index kept memory-mapped,
for traversal. But a quantized table also writes a `.rawf32` companion: the exact vectors
the rescore reads. It is roughly the corpus's uncompressed size, so the whole bundle is not
smaller than the `f32` table's.

In [ ]:
int8_total = int8_bytes["usearch"] + int8_bytes["rawf32"]
print(f"f32 bundle:  {sum(f32_bytes.values()):,} bytes")
print(f"int8 bundle: {sum(int8_bytes.values()):,} bytes  (graph + companion {int8_total:,})")

So the precise claim is about the **graph**: `storage_precision=int8` shrinks the structure
a search traverses, not the bytes committed to disk. The companion is a flat array a search
touches at only `k * oversample` offsets per query, not something traversal walks.

## Dialing `oversample` — the recall/cost operating point

`oversample` is the retrieve stage's candidate breadth: a candidate the rescore never sees
cannot be recovered, however exact the rescore. Four otherwise-identical `int8` tables, each
stamped with a different table-default `oversample`:

In [ ]:
GRID = [1, 2, 4, 8]
sweep = {}
for ov in GRID:
    with precision.built(corpus, "int8", ov) as (db, _):
        sweep[ov] = precision.recall(db, corpus)
print(f"{'oversample':>10}{'candidates':>12}{'recall@10':>12}")
for ov in GRID:
    print(f"{ov:>10}{K * ov:>12}{sweep[ov]:>12.4f}")

In [ ]:
for ov in GRID:
    contracts.assert_close(f"precision.int8_os{ov}_sweep_recall_at_10", sweep[ov], tol=0.02)
assert all(sweep[a] <= sweep[b] + 1e-9 for a, b in zip(GRID, GRID[1:])), sweep
assert sweep[1] < sweep[4], "a narrow candidate set loses true neighbours"

Recall never falls as the candidate set widens. The narrowest setting, `oversample=1` — as
many candidates as results — loses true neighbours the quantized graph ranked just outside
the top `k`, which no rescore downstream can recover; widening recovers them, and by the
engine's default of `4` recall has reached the exact baseline. The default is where the
curve flattens, not an arbitrary constant.

## The per-request override — `search(..., oversample=...)`

`oversample` also has a per-request form, resolved *request > table default > deployment
default*. On the table stamped at `oversample=1`, the same queries with an explicit
per-request `oversample=8`:

In [ ]:
with precision.built(corpus, "int8", 1) as (db, _):
    table_default = precision.recall(db, corpus)
    explicit_one = precision.recall(db, corpus, oversample=1)
    widened = precision.recall(db, corpus, oversample=8)
print(f"table default (1):          recall@{K} = {table_default:.4f}")
print(f"explicit per-request 1:     recall@{K} = {explicit_one:.4f}")
print(f"per-request override to 8:  recall@{K} = {widened:.4f}")

In [ ]:
assert table_default == explicit_one, "no override resolves to the table's own default"
assert widened >= table_default
assert widened == sweep[8], "a per-request 8 is the table-default 8's search"
contracts.assert_close("precision.override8_on_os1_recall_at_10", widened, tol=0.02)

An override recovers exactly the recall a wider table default gives, on the same on-disk
graph, with no rebuild — reachable where a caller looks for it, `search`'s own keywords.

## Bridge note

> **Quantization is a real recall trade, and rescore is what makes the trade honest.** HNSW
> (Malkov & Yashunin 2020) gives approximate search its speed by navigating a proximity graph instead
> of scanning every vector; scalar quantization shrinks that graph further by storing its own
> vectors at lower precision, which is exactly the kind of accuracy-for-memory trade billion-scale
> ANN systems live or die on (Johnson et al. 2021). The engine's two-stage retrieve→rescore is
> what keeps that trade from silently degrading the *answer*: the graph proposes from lossy
> vectors, but the response is always re-derived from the exact ones — so quantization buys
> memory on the structure a search traverses without buying error into what a caller receives,
> *provided* `oversample` is wide enough that the true neighbours are still in the candidate set
> the rescore gets to see. Measured here at `oversample=4` (the engine's own default), that
> provision holds exactly: `int8` recall matches `f32` recall while the graph shrinks to
> roughly a quarter its size — the trade this book keeps insisting on: not assumed, measured.
> And both knobs — the deployment-default `storage_precision` and the per-request
> `oversample` override — are reachable from the same public `jammi.connect` / `search` surface
> every other chapter in this book writes against.

## References

- Johnson, Jeff, Douze, Matthijs, Jégou, Hervé (2021) *Billion-Scale Similarity Search with GPUs* IEEE Transactions on Big Data DOI 10.1109/TBDATA.2019.2921572; arXiv:1702.08734.
- Malkov, Yu A., Yashunin, Dmitry A. (2020) *Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs* IEEE Transactions on Pattern Analysis and Machine Intelligence DOI 10.1109/TPAMI.2018.2889473; arXiv:1603.09320.